In [204]:
import pandas as pd
import duckdb
import os
import os, json
from uuid import uuid4
import pandas as pd
pd.set_option("display.max_colwidth", 200)
#pd.set_option("display.width", 500)  # adjusts total line width before wrapping
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer
import faiss
from sklearn.preprocessing import normalize
import gspread
from gspread_dataframe import get_as_dataframe, set_with_dataframe
from google.oauth2 import service_account # based on google-auth library
import matplotlib.pyplot as plt
import re
import unicodedata
from sklearn.preprocessing import normalize
import numpy as np
import json
from sklearn.preprocessing import normalize
import numpy as np
import json
import math
from collections import defaultdict
from sklearn.preprocessing import normalize
import Levenshtein


In [2]:
try:
    file_data = json.load(open(os.path.expanduser("~/ServiceAccountsKey.json")))
    # (2) transform the content into crendentials object
    credentials = service_account.Credentials.from_service_account_info(file_data)
# (3) specify your usage of the credentials
    scoped_credentials = credentials.with_scopes(['https://spreadsheets.google.com/feeds', 'https://www.googleapis.com/auth/drive'])
# (4) use the constrained credentials for authentication of gspread package
    gc = gspread.Client(auth=scoped_credentials)
    grela_gs = gc.open_by_url("https://docs.google.com/spreadsheets/d/1QroTEQ9gQf9cLO9mvolp7fELNbYYjgvGd48yAiTj03w/edit?usp=sharing")
except:
    pass

In [3]:
model = SentenceTransformer("julian-schelb/multilingual-e5-large-emb-lat-intertext-v1")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [61]:
vulgate_df = pd.read_parquet("../data/large_files/vulgate_df.parquet")

In [62]:
emb_matrix = np.load("../data/large_files/vulgate_embeddings.npz", allow_pickle=True)["embeddings"]

In [63]:
d = emb_matrix.shape[1]  # 768
index = faiss.IndexFlatIP(d)        # Inner product = cosine if normalized
index.add(emb_matrix)

In [64]:
register_df = pd.read_parquet("../data/large_files/register_df_with_embeddings.parquet")

In [65]:
verse_index = vulgate_df[
    (vulgate_df["title"] == "Matthew") &
    (vulgate_df["chapter"] == "16") &
    (vulgate_df["verse"] == "18")
].index[0]
verse_emb = emb_matrix[verse_index]#.to_dict(orient="records")

In [159]:
verse_index

566

In [71]:
verse_text = vulgate_df.iloc[verse_index]["sent_text_clean"]
verse_text

'et ego dico tibi quia tu es petrus et super hanc petram aedificabo ecclesiam meam et portae inferi non praeualebunt aduersum eam'

In [11]:
vulgate_df["sem_lemmata_string"] = vulgate_df["tokens"].apply(lambda tokens: [t["lemma"] for t in tokens if t["pos"] in ["NOUN", "ADJ", "VERB", "PROPN"]])

In [ ]:
sentence = 'dominus enim iesus christus beatum petrum constituit principem apostolorum dans ei claues regni coelorum et potestatem ligandi et soluendi in coelo et in terra super quem etiam ecclesiam suam aedificauit commendans ei oues suas pascendas ex quo tempore principatus ille et potestas per beatum petrum successit omnibus suam cathedram suscipientibus uel usque in finem mundi suscepturis diuino priuilegio et iure haereditario'

In [36]:
def preprocess_string(string):
    # Normalize Unicode (optional, for accented chars)
    string = unicodedata.normalize("NFD", string)
    # Remove all non-alphabetic characters (keep spaces)
    string = re.sub(r"[^a-zA-Z\s]", "", string)
    # Lowercase and Latin orthographic normalization
    string = string.lower().replace("v", "u").replace("j", "i")
    return string



'dsdsdsese uuewew'

In [125]:
def jaccard_sim(string1, string2):
    set1 = set(string1.split())
    set2 = set(string2.split())
    union = set1 | set2
    if not union:
        return 0.0
    return len(set1 & set2) / len(union)

def levenshtein_sim(string1, string2):
    return Levenshtein.ratio(string1, string2)

def embedding_sim(string1, string2, model=model):
    embs = model.encode([string1, string2], convert_to_numpy=True)
    # cosine similarity
    return np.dot(embs[0], embs[1].T).flatten()[0]

def measure_similarity(string1, string2, model=model):
    #string1 = preprocess_string(string1)
    #string2 = preprocess_string(string2)
    return {"string1" : string1,
            "string2" : string2,
            "jaccard_sim": jaccard_sim(string1, string2),
            "levenshtein_sim": levenshtein_sim(string1, string2),
            "embedding_sim" : embedding_sim(string1, string2, model=model)}


In [134]:
string1 = "He works at a university at the Czech Republic."
string2 = "He is an employee of an academic institution in Central Europe"
df = pd.DataFrame([measure_similarity(string1, string2)])
df

,string1,string2,jaccard_sim,levenshtein_sim,embedding_sim
0,He works at a university at the Czech Republic.,He is an employee of an academic institution in Central Europe,0.058824,0.366972,0.848621


In [127]:
set_with_dataframe(grela_gs.add_worksheet("example1", 1,1), df)

In [130]:
string1 = "It shows her sitting on a river bank."
string2 = "It shows her sitting in a private bank."
df = pd.DataFrame([measure_similarity(string1, string2)])
df

,string1,string2,jaccard_sim,levenshtein_sim,embedding_sim
0,It shows her sitting on a river bank.,It shows her sitting in a private bank.,0.6,0.921053,0.793461


In [131]:
set_with_dataframe(grela_gs.add_worksheet("example2", 1,1), df)

In [132]:
string1 = "He works at a university in the Czech Republic."
string2 = "Er arbeitet an einer Hochschule in Tschechien."
df = pd.DataFrame([measure_similarity(string1, string2)])
df

,string1,string2,jaccard_sim,levenshtein_sim,embedding_sim
0,He works at a university in the Czech Republic.,Er arbeitet an einer Hochschule in Tschechien.,0.066667,0.451613,0.884416


In [133]:
set_with_dataframe(grela_gs.add_worksheet("example3", 1,1), df)


In [38]:
tokens = register_df[register_df["sent_text_clean"].str.contains("dominus enim iesus christus")]["tokens"].values[0]
tokens

array([{'char_end': 7, 'char_start': 0, 'lemma': 'Dominus', 'pos': 'NOUN', 'register_ref': '361;4', 'token_id': 176903814, 'token_text': 'Dominus'},
       {'char_end': 12, 'char_start': 8, 'lemma': 'enim', 'pos': 'ADV', 'register_ref': '361;4', 'token_id': 176903815, 'token_text': 'enim'},
       {'char_end': 18, 'char_start': 13, 'lemma': 'Jesus', 'pos': 'NOUN', 'register_ref': '361;4', 'token_id': 176903816, 'token_text': 'Jesus'},
       {'char_end': 27, 'char_start': 19, 'lemma': 'Christus', 'pos': 'NOUN', 'register_ref': '361;4', 'token_id': 176903817, 'token_text': 'Christus'},
       {'char_end': 34, 'char_start': 28, 'lemma': 'beatus', 'pos': 'ADJ', 'register_ref': '361;4', 'token_id': 176903818, 'token_text': 'beatum'},
       {'char_end': 41, 'char_start': 35, 'lemma': 'Petrus', 'pos': 'NOUN', 'register_ref': '361;4', 'token_id': 176903819, 'token_text': 'Petrum'},
       {'char_end': 52, 'char_start': 42, 'lemma': 'constituo', 'pos': 'VERB', 'register_ref': '361;4', 'token_

In [210]:
def build_ngrams_df(tokens):
    ngrams = []

    # first add full sentence
    tokens_length = len(tokens)

    min_window_size = 5
    max_window_size = tokens_length - 1

    ngram_tokens_data = tokens
    ngram_tokens_clean = " ".join([preprocess_string(t["token_text"]) for t in ngram_tokens_data])
    ngram_lemmata_clean =  " ".join([preprocess_string(t["lemma"]) for t in ngram_tokens_data if t["pos"] not in ["PUNCT", "SPACE"]])
    ngrams.append({
            "ngram_tokens" : ngram_tokens_clean,
            "ngram_lemmata" : ngram_lemmata_clean,
            "ngram_start" : 0,
            "ngram_stop" : tokens_length,
            "length" : tokens_length})


    for n in range(min_window_size, max_window_size + 1):
        # let's build gradually longer ngrams from the sentence beginnings, starting from 5 to 20 tokens
        ngram_tokens_data = tokens[:n]
        ngram_tokens_clean = " ".join([preprocess_string(t["token_text"]) for t in ngram_tokens_data])
        ngram_lemmata_clean =  " ".join([preprocess_string(t["lemma"]) for t in ngram_tokens_data if t["pos"] not in ["PUNCT", "SPACE"]])
        ngrams.append({
            "ngram_tokens" : ngram_tokens_clean,
            "ngram_lemmata" : ngram_lemmata_clean,
            "ngram_start" : 0,
            "ngram_stop" : n,
            "length" : len(ngram_tokens_data)})

    for n in range(min_window_size, max_window_size + 1):
        # let's build gradually longer ngrams from the sentence beginnings, starting from 5 to 20 tokens
        ngram_tokens_data = tokens[-n:]
        ngram_tokens_clean = " ".join([preprocess_string(t["token_text"]) for t in ngram_tokens_data])
        ngram_lemmata_clean =  " ".join([preprocess_string(t["lemma"]) for t in ngram_tokens_data if t["pos"] not in ["PUNCT", "SPACE"]])
        ngrams.append({
            "ngram_tokens" : ngram_tokens_clean,
            "ngram_lemmata" : ngram_lemmata_clean,
            "ngram_start" : tokens_length - n,
            "ngram_stop" : tokens_length,
            "length" : len(ngram_tokens_data)
        })
    ngrams_df = pd.DataFrame(ngrams)
    return ngrams_df

In [233]:
ngrams_df = build_ngrams_df(tokens)

In [213]:
ngram_vecs =  model.encode(ngrams_df["ngram_tokens"].tolist(), convert_to_numpy=True)
ngram_vecs = normalize(ngram_vecs, norm='l2')

In [178]:
# test with one verse
# scores = np.dot(ngram_vecs, verse_emb.T).flatten()
# ngrams_df["emb"] = scores


In [248]:
def score_ngram_matches(
    ngrams_df,
    model,
    index,
    vulgate_df,
    text_col="ngram_tokens",
    vulgate_text_col="sent_text_clean",
    top_k=10,
):
    # embeddings for ngrams
    ngram_vecs = model.encode(ngrams_df[text_col].tolist(), convert_to_numpy=True)
    ngram_vecs = normalize(ngram_vecs, norm="l2")

    # retrieve nearest Vulgate rows
    scores, indices = index.search(ngram_vecs, top_k)

    indices_list = []
    emb_scores_list = []
    jaccard_scores_list = []
    levenshtein_scores_list = []
    vulgate_lens_list = []
    lenratio_scores_list = []

    for row_idx in range(len(ngrams_df)):
        ngram_text = ngrams_df.iloc[row_idx][text_col]

        row_indices = []
        row_emb_scores = []
        row_jaccard_scores = []
        row_lev_scores = []
        row_vulgate_lens = []
        row_lenratio_scores = []

        for i, score in zip(indices[row_idx], scores[row_idx]):
            if i < 0:
                continue

            vulgate_text = preprocess_string(vulgate_df.iloc[i][vulgate_text_col])
            j_score = jaccard_sim(ngram_text, vulgate_text)
            l_score = levenshtein_sim(ngram_text, vulgate_text)

            row_indices.append(int(i))
            row_emb_scores.append(float(score))
            row_jaccard_scores.append(j_score)
            row_lev_scores.append(l_score)

            vulgate_len = len(vulgate_text.split())
            row_vulgate_lens.append(vulgate_len)
            if vulgate_len > 0:
                row_lenratio_scores.append(len(ngram_text.split()) / vulgate_len)
            else:
                row_lenratio_scores.append(0.0)

        indices_list.append(row_indices)
        emb_scores_list.append(row_emb_scores)
        jaccard_scores_list.append(row_jaccard_scores)
        levenshtein_scores_list.append(row_lev_scores)
        vulgate_lens_list.append(row_vulgate_lens)
        lenratio_scores_list.append(row_lenratio_scores)
    return pd.DataFrame({
        "indices": indices_list,
        "embedding_scores": emb_scores_list,
        "jaccard_scores": jaccard_scores_list,
        "levenshtein_scores": levenshtein_scores_list,
        "vulgate_lens" : vulgate_lens_list,
        "lenratio_scores" : lenratio_scores_list,
    })

In [250]:
ngrams_df[[
    "indices",
    "embedding_scores",
    "jaccard_scores",
    "levenshtein_scores",
    "vulgate_lens",
    "lenratio_scores",
]] = score_ngram_matches(
    ngrams_df=ngrams_df,
    model=model,
    index=index,
    vulgate_df=vulgate_df,
    top_k=10,
)

In [251]:
ngrams_df

,ngram_tokens,ngram_lemmata,ngram_start,ngram_stop,length,indices,embedding_scores,jaccard_scores,levenshtein_scores,lenratio_scores,vulgate_lens
0,dominus enim iesus christus beatum petrum constituit principem apostolorum dans ei claues regni coelorum et potestatem ligandi et soluendi in coelo et in terra super quem etiam ecclesiam suam aedi...,dominus enim iesus christus beatus petrus constituo princeps apostolus do is claues regnum coelum et potestas ligo et soluo in coelum et in terra super qui etiam ecclesia suus aedifico commendo is...,0,61,61,"[6325, 13271, 23956, 6323, 19782, 29702, 28778, 3973, 6817, 7225]","[0.7533336877822876, 0.7478184103965759, 0.7449114322662354, 0.7405662536621094, 0.7398617267608643, 0.7389773726463318, 0.7346551418304443, 0.734064519405365, 0.7337177991867065, 0.7330724000930786]","[0.0625, 0.0684931506849315, 0.05263157894736842, 0.07575757575757576, 0.037037037037037035, 0.05, 0.04477611940298507, 0.029850746268656716, 0.03225806451612903, 0.015873015873015872]","[0.33641404805914976, 0.39095315024232635, 0.2677484787018256, 0.43234323432343236, 0.41520467836257313, 0.25844930417495027, 0.3533697632058288, 0.33395176252319114, 0.2846153846153846, 0.3074003...","[3.210526315789474, 2.103448275862069, 6.1, 2.033333333333333, 1.4523809523809523, 5.083333333333333, 3.05, 3.388888888888889, 4.6923076923076925, 4.6923076923076925]","[19, 29, 10, 30, 42, 12, 20, 18, 13, 13]"
1,dominus enim iesus christus beatum,dominus enim iesus christus beatus,0,5,5,"[7889, 6541, 5618, 5764, 6498, 5969, 6119, 6213, 6501, 6058]","[0.8409472107887268, 0.8284190893173218, 0.823582112789154, 0.8222182393074036, 0.8219188451766968, 0.820804238319397, 0.8182109594345093, 0.8159699440002441, 0.815969705581665, 0.815969705581665]","[0.0, 0.0, 0.1111111111111111, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[0.6075949367088608, 0.5777777777777777, 0.6, 0.36764705882352944, 0.5777777777777777, 0.4, 0.4247787610619469, 0.4, 0.4, 0.4]","[0.7142857142857143, 0.5555555555555556, 0.8333333333333334, 0.35714285714285715, 0.625, 0.4166666666666667, 0.45454545454545453, 0.4166666666666667, 0.4166666666666667, 0.4166666666666667]","[7, 9, 6, 14, 8, 12, 11, 12, 12, 12]"
2,dominus enim iesus christus beatum petrum,dominus enim iesus christus beatus petrus,0,6,6,"[7889, 5764, 5969, 5963, 6541, 6801, 6501, 6058, 5653, 5327]","[0.8294388651847839, 0.8224636316299438, 0.8212431073188782, 0.8183884620666504, 0.8182430267333984, 0.8169776797294617, 0.8169776797294617, 0.8169776797294617, 0.8169776797294617, 0.8169776797294...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[0.5813953488372092, 0.4055944055944056, 0.37254901960784315, 0.5471698113207547, 0.5567010309278351, 0.37254901960784315, 0.37254901960784315, 0.37254901960784315, 0.37254901960784315, 0.37254901...","[0.8571428571428571, 0.42857142857142855, 0.5, 0.6, 0.6666666666666666, 0.5, 0.5, 0.5, 0.5, 0.5]","[7, 14, 12, 10, 9, 12, 12, 12, 12, 12]"
3,dominus enim iesus christus beatum petrum constituit,dominus enim iesus christus beatus petrus constituo,0,7,7,"[32223, 5764, 7330, 3760, 24628, 6118, 16510, 6163, 7242, 20354]","[0.7837698459625244, 0.7729225158691406, 0.7712094783782959, 0.7695132493972778, 0.7682401537895203, 0.7629792094230652, 0.7627385258674622, 0.7626582384109497, 0.7624891996383667, 0.7624440789222...","[0.07692307692307693, 0.0, 0.0, 0.0, 0.0, 0.0, 0.06666666666666667, 0.0, 0.0, 0.11764705882352941]","[0.4807692307692307, 0.48051948051948057, 0.37696335078534027, 0.4385964912280702, 0.4380952380952381, 0.41666666666666663, 0.485981308411215, 0.40944881889763785, 0.3906976744186047, 0.4296296296...","[0.875, 0.5, 0.3181818181818182, 0.6363636363636364, 0.7777777777777778, 0.5384615384615384, 0.6363636363636364, 0.7, 0.2916666666666667, 0.5384615384615384]","[8, 14, 22, 11, 9, 13, 11, 10, 24, 13]"
4,dominus enim iesus christus beatum petrum constituit principem,dominus enim iesus christus beatus petrus constituo princeps,0,8,8,"[32223, 3760, 6118, 7330, 3791, 10

In [252]:
def from_tokens_to_jsons(tokens, sentence_id):
    ngrams_df = build_ngrams_df(tokens)
    ngrams_df[[
        "indices",
        "embedding_scores",
        "jaccard_scores",
        "levenshtein_scores",
        "vulgate_lens",
        "lenratio_scores",
    ]] = score_ngram_matches(
        ngrams_df=ngrams_df,
        model=model,
        index=index,
        vulgate_df=vulgate_df,
        text_col="ngram_tokens",
        vulgate_text_col="sent_text_clean",
        top_k=10,
    )
    ngrams_df.to_json("../data/ngrams_dfs/{}.json".format(sentence_id))

In [253]:
# test with a sample
register_df.sample(10, random_state=0).apply(lambda row: from_tokens_to_jsons(row["tokens"], row["sentence_id"]), axis=1)

1454    None
3119    None
3817    None
469     None
3025    None
2937    None
944     None
4264    None
4848    None
4306    None
dtype: object

In [254]:
%%time
register_df.apply(lambda row: from_tokens_to_jsons(row["tokens"], row["sentence_id"]), axis=1)

CPU times: user 2h 13min 29s, sys: 33.5 s, total: 2h 14min 2s
Wall time: 29min 2s


0       None
1       None
1112    None
2223    None
3334    None
        ... 
4884    None
4885    None
4886    None
4887    None
4888    None
Length: 5399, dtype: object

In [238]:
vulgate_df["sent_text_clean"]

0                                                                                                                                                      liber generationis iesu christi filii dauid filii abraham
1                                                                                                                         abraham genuit isaac isaac autem genuit iacob iacob autem genuit iudam et fratres eius
2                                                                                                                                     iosias autem genuit iechoniam et fratres eius in transmigratione babylonis
3                                                                                          beati estis cum maledixerint uobis et persecuti uos fuerint et dixerint omne malum aduersum uos mentientes propter me
4                                                                                                         congregatis ergo illis dixit pilatus quem uultis dimittam 

In [240]:
vulgate_df[vulgate_df["sent_text_clean"].apply(lambda x: len(x.split())) == 0]

,sentence_id,grela_id,sentence_position,sent_text,author,title,tokens,chapter,verse,sent_text_clean
833,vulgate_tlg0031.tlg001.obi-lat_786,vulgate_tlg0031.tlg001.obi-lat,786,[ ],Novum Testamentum,Matthew,"[{'chapter': '23', 'char_end': 1, 'char_start': 0, 'lemma': '[', 'pos': 'X', 'token_id': 255109220, 'token_text': '[', 'verse': '14'}, {'chapter': '23', 'char_end': 3, 'char_start': 2, 'lemma': ']...",23,14,
2976,vulgate_tlg0031.tlg004.obi-lat_169,vulgate_tlg0031.tlg004.obi-lat,169,[ ],Johnannine literature,John,"[{'chapter': '5', 'char_end': 1, 'char_start': 0, 'lemma': '[', 'pos': 'X', 'token_id': 255347153, 'token_text': '[', 'verse': '4'}, {'chapter': '5', 'char_end': 3, 'char_start': 2, 'lemma': ']', ...",5,4,
3783,vulgate_tlg0031.tlg005.obi-lat_1001,vulgate_tlg0031.tlg005.obi-lat,1001,[ ],Luke (the evangelist),Acts,"[{'chapter': '28', 'char_end': 1, 'char_start': 0, 'lemma': '[', 'pos': 'X', 'token_id': 255017914, 'token_text': '[', 'verse': '29'}, {'chapter': '28', 'char_end': 3, 'char_start': 2, 'lemma': ']...",28,29,
3992,vulgate_tlg0031.tlg005.obi-lat_288,vulgate_tlg0031.tlg005.obi-lat,288,[ ],Luke (the evangelist),Acts,"[{'chapter': '8', 'char_end': 1, 'char_start': 0, 'lemma': '[', 'pos': 'X', 'token_id': 255006058, 'token_text': '[', 'verse': '37'}, {'chapter': '8', 'char_end': 3, 'char_start': 2, 'lemma': ']',...",8,37,
4002,vulgate_tlg0031.tlg005.obi-lat_297,vulgate_tlg0031.tlg005.obi-lat,297,[ ],Luke (the evangelist),Acts,"[{'chapter': '9', 'char_end': 1, 'char_start': 0, 'lemma': '[', 'pos': 'X', 'token_id': 255006187, 'token_text': '[', 'verse': '6'}, {'chapter': '9', 'char_end': 3, 'char_start': 2, 'lemma': ']', ...",9,6,
4284,vulgate_tlg0031.tlg005.obi-lat_550,vulgate_tlg0031.tlg005.obi-lat,550,[ ],Luke (the evangelist),Acts,"[{'chapter': '15', 'char_end': 1, 'char_start': 0, 'lemma': '[', 'pos': 'X', 'token_id': 255010330, 'token_text': '[', 'verse': '34'}, {'chapter': '15', 'char_end': 3, 'char_start': 2, 'lemma': ']...",15,34,
4378,vulgate_tlg0031.tlg005.obi-lat_635,vulgate_tlg0031.tlg005.obi-lat,635,[ ],Luke (the evangelist),Acts,"[{'chapter': '18', 'char_end': 1, 'char_start': 0, 'lemma': '[', 'pos': 'X', 'token_id': 255011745, 'token_text': '[', 'verse': '4'}, {'chapter': '18', 'char_end': 3, 'char_start': 2, 'lemma': ']'...",18,4,
4597,vulgate_tlg0031.tlg005.obi-lat_832,vulgate_tlg0031.tlg005.obi-lat,832,[ ],Luke (the evangelist),Acts,"[{'chapter': '23', 'char_end': 1, 'char_start': 0, 'lemma': '[', 'pos': 'X', 'token_id': 255015113, 'token_text': '[', 'verse': '25'}, {'chapter': '23', 'char_end': 3, 'char_start': 2, 'lemma': ']...",23,25,
4615,vulgate_tlg0031.tlg005.obi-lat_849,vulgate_tlg0031.tlg005.obi-lat,849,[ ],Luke (the evangelist),Acts,"[{'chapter': '24', 'char_end': 1, 'char_start': 0, 'lemma': '[', 'pos': 'X', 'token_id': 255015345, 'token_text': '[', 'verse': '7'}, {'chapter': '24', 'char_end': 3, 'char_start': 2, 'lemma': ']'...",24,7,
5149,vulgate_tlg0031.tlg006.obi-lat_429,vulgate_tlg0031.tlg006.obi-lat,429,[ ],Paul of Tarsus,Romans,"[{'chapter': '16', 'char_end': 1, 'char_start': 0, 'lemma': '[', 'pos': 'X', 'token_id': 255377074, 'token_text': '[', 'verse': '24'}, {'chapter': '16', 'char_end': 3, 'char_start': 2, 'lemma': ']...",16,24,


In [ ]:
def detect_multiple_vulgate_matches_with_ngrams(
    sentence,
    model,
    index,
    vulgate_df,
    top_k_per_ngram=3,
    min_sim=0.75,
    max_total=5,
    window_size=10,
    step=1,
    include_full_sentence=True,
):
    """
    Search for likely Vulgate parallels by embedding overlapping fixed-size
    windows from `sentence` and keeping the best-scoring match per Vulgate verse.
    Returns only compact match records.
    """
    from collections import defaultdict
    from sklearn.preprocessing import normalize

    sentence = sentence.strip()
    if not sentence:
        return []

    tokens = sentence.split()
    n_tokens = len(tokens)
    if n_tokens == 0:
        return []

    window_size = min(window_size, n_tokens)

    windows = []
    for start in range(0, n_tokens - window_size + 1, step):
        end = start + window_size
        windows.append({
            "text": " ".join(tokens[start:end]),
            "start": start,
            "end": end,
        })

    if include_full_sentence and n_tokens > window_size:
        windows.append({
            "text": sentence,
            "start": 0,
            "end": n_tokens,
        })

    if not windows:
        windows = [{
            "text": sentence,
            "start": 0,
            "end": n_tokens,
        }]

    window_texts = [w["text"] for w in windows]
    window_vecs = model.encode(window_texts, convert_to_numpy=True)
    window_vecs = normalize(window_vecs, norm="l2")

    hits = {}

    for window, vec in zip(windows, window_vecs):
        scores, indices = index.search(vec.reshape(1, -1), top_k_per_ngram)

        for score, idx in zip(scores[0], indices[0]):
            if idx < 0 or score < min_sim:
                continue

            row = vulgate_df.iloc[idx]
            sid = row["sentence_id"]

            if sid not in hits or score > hits[sid]["best_raw_score"]:
                hits[sid] = {
                    "best_raw_score": float(score),
                    "best_ngram": window["text"],
                    "vulgate_text_clean": row["sent_text_clean"],
                    "author": row["author"],
                    "title": row["title"],
                    "chapter": row["chapter"],
                    "verse": row["verse"],
                }

    results = sorted(hits.values(), key=lambda x: x["best_raw_score"], reverse=True)

    if max_total is not None:
        results = results[:max_total]

    return results

In [ ]:
# Collapse the ngram level to per-sentence matches:
# one compact match record per distinct Vulgate verse (best-scoring ngram),
# ranked by best_raw_score, up to max_total per sentence.
register_df["matches"] = register_df["sent_text_clean"].apply(
    lambda sentence: detect_multiple_vulgate_matches_with_ngrams(
        sentence,
        model,
        index,
        vulgate_df,
        top_k_per_ngram=3,
        min_sim=0.75,
        max_total=5,
        window_size=10,
        step=1,
        include_full_sentence=True,
    )
)

In [ ]:
matches_rows = []

for _, row in register_df.iterrows():
    for match_rank, match in enumerate(row["matches"], start=1):
        matches_rows.append({
            "source_sentence_id": row["sentence_id"],
            "register_ref": row["register_ref"],
            "source_sentence": row["sent_text_clean"],
            "match_rank_within_sentence": match_rank,
            "best_ngram": match["best_ngram"],
            "vulgate_text_clean": match["vulgate_text_clean"],
            "author": match["author"],
            "title": match["title"],
            "chapter": match["chapter"],
            "verse": match["verse"],
            "best_raw_score": match["best_raw_score"],
        })

matches_long_df = pd.DataFrame(matches_rows)

In [ ]:
matches_long_df = matches_long_df[
    [
        "source_sentence_id",
        "register_ref",
        "source_sentence",
        "best_ngram",
        "vulgate_text_clean",
        "author",
        "title",
        "chapter",
        "verse",
        "best_raw_score",
    ]
]
matches_long_df.head(5)

In [ ]:
# Combined score: mean of the raw embedding score, Jaccard, and Levenshtein similarities.
# Recovered from register_ngram_matches_v4.parquet (verified 1:1 — matches the saved combined_score exactly).
matches_long_df["jaccard_sim"] = matches_long_df.apply(
    lambda r: jaccard_sim(r["best_ngram"], r["vulgate_text_clean"]), axis=1
)
matches_long_df["levenshtein_sim"] = matches_long_df.apply(
    lambda r: levenshtein_sim(r["best_ngram"], r["vulgate_text_clean"]), axis=1
)
matches_long_df["combined_score"] = (
    matches_long_df["best_raw_score"]
    + matches_long_df["jaccard_sim"]
    + matches_long_df["levenshtein_sim"]
) / 3

matches_long_df.head(5)

In [ ]:
sum(matches_long_df["best_raw_score"] >= 0.8)

In [ ]:
matches_per_sentence = matches_long_df.groupby("source_sentence_id").size().sort_values(ascending=False)
matches_per_sentence.max()

In [94]:
matches_long_df.head(5)

,source_sentence_id,register_ref,source_sentence,best_ngram,vulgate_text_clean,author,title,chapter,verse,best_raw_score,jaccard_sim,levenshtein_distance,levenshtein_sim,combined_score
0,cc_10265_4,1;3,rogat ut deum pro se deprecetur et ad se quantocius ueniat,rogat ut deum pro se deprecetur et ad se quantocius,ad te domine clamabo et ad deum meum deprecabor,NaN,Psalmi,29,9,0.821929,0.214286,0.571429,0.428571,0.488262
1,cc_10265_4,1;3,rogat ut deum pro se deprecetur et ad se quantocius ueniat,rogat ut deum pro se deprecetur et ad se quantocius,quam ob rem ego deprecabor dominum et ad deum ponam eloquium meum,NaN,Job,5,8,0.801750,0.166667,0.517241,0.482759,0.483725
2,cc_10265_4,1;3,rogat ut deum pro se deprecetur et ad se quantocius ueniat,ut deum pro se deprecetur et ad se quantocius ueniat,propterea exaudiuit deus adtendit uoci deprecationis meae,NaN,Psalmi,65,19,0.798879,0.000000,0.559633,0.440367,0.413082
3,cc_10265_4,1;3,rogat ut deum pro se deprecetur et ad se quantocius ueniat,rogat ut deum pro se deprecetur et ad se quantocius,qui rogabit pro eo coram domino et dimittetur illi pro singulis quae faciendo peccauerit,NaN,Leviticus,6,7,0.779364,0.100000,0.553957,0.446043,0.441802
4,cc_10265_5,1;4,gregorius in romanum pontificem electus desiderio abbati monasterii sancti benedicti montis cassini salutem in christo iesu,abbati monasterii sancti benedicti montis cassini salutem in christo iesu,salutate fratres omnes in osculo sancto,Paul of Tarsus,1 Thessalonians,5,26,0.770857,0.066667,0.571429,0.428571,0.422032


In [95]:
matches_long_df.to_parquet("../data/register_ngram_matches_v4.parquet")

In [68]:
set_with_dataframe(grela_gs.add_worksheet("grela_register_ngram_matches_v4", 1,1), matches_long_df[matches_long_df["best_raw_score"]>=0.8])